## World Map: Heatmap KDE Density Plot

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
from pathlib import Path
from eda_toolkit import generate_table1

In [ ]:
df_eda_geocoded = pd.read_parquet("../data/processed/df_eda_geocoded.parquet")

In [ ]:
# Define File Paths for Files inside /geo_maps/world
geo_maps_dir_world = Path("../geo_maps/world")
matches = list(geo_maps_dir_world.rglob("ne_110m_admin_0_countries.shp"))

if not matches:
    # Fallback search for any .shp file if specific filename isn't matched
    matches = list(geo_maps_dir_world.rglob("*.shp"))

if matches:
    world_path = str(matches[0])
    print(f"Loading shapefile from: {world_path}")
    world = gpd.read_file(world_path)
else:
    raise FileNotFoundError(f"No .shp files found in {geo_maps_dir_world}")

# Build sighting GeoDataFrame
gdf = gpd.GeoDataFrame(
    df_eda_geocoded,
    geometry=gpd.points_from_xy(df_eda_geocoded["longitude"], df_eda_geocoded["latitude"]),
    crs="EPSG:4326",  # Set Standard Coordinate Reference System (CRS) w/ lat/lonn
)

# If there's no access to a .prj file: explicitly assign CRS (EPSG:4326)
world = world.set_crs("EPSG:4326", allow_override=True)

# Plot the KDE Heatmap (Sightings Gradient)
fig, ax = plt.subplots(figsize=(10, 8))

world.plot(ax=ax, color="#e0e0e0", edgecolor="white", linewidth=0.5)

# Overlay the KDE based on sighting positions
sns.kdeplot(
    x=gdf.geometry.x,
    y=gdf.geometry.y,
    cmap="YlOrRd",  # Yellow-Orange-Red gradient color scheme
    fill=True,
    alpha=0.6,
    levels=15,  # Smoothness/resolution of density contours
    ax=ax,
)

# Plot individual sighting points on top
gdf.plot(
    ax=ax,
    color="black",
    markersize=8,
    alpha=0.7,
    label="Individual Sightings",
)

# Framing & Formatting
minx, miny, maxx, maxy = gdf.total_bounds
buffer = 1.0  # Buffer in degrees
ax.set_xlim(minx - buffer, maxx + buffer)
ax.set_ylim(miny - buffer, maxy + buffer)

ax.set_title(
    f"Sightings Heatmap (Total Reports: {len(gdf)})",
    fontsize=14,
    fontweight="bold",
)
ax.set_axis_off()
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

## US Map 1: Heatmap KDE Density Plot

In [ ]:
# Define File Paths for Files inside /geo_maps/us
us_dir = Path("../geo_maps/us")

# Paths to the US State base map & US State shapefiles
us_states_path = us_dir / "tl_2023_us_state.shp"

# Load both shapefile layers
us_states = gpd.read_file(us_states_path)

# Build Sighting GeoDataFrame & Align CRS
gdf = gpd.GeoDataFrame(
    df_eda_geocoded,
    geometry=gpd.points_from_xy(df_eda_geocoded["longitude"], df_eda_geocoded["latitude"]),
    crs="EPSG:4326",  # Set Standard Coordinate Reference System (CRS) w/ lat/lon
)

# Align CRS for the shapefile to match sightings data
us_states = us_states.to_crs(gdf.crs)

# Filter sighting data down to Continental US (CONUS)
us_gdf = gdf.cx[-125:-66, 24:50]

# Build Layered US Map
plt.close("all")  # Prevent plot memory overlap
fig, ax = plt.subplots(figsize=(14, 10))

# Layer 1: US State Boundaries Base Map (Light gray fill w/ white state borders)
us_states.plot(ax=ax, color="#e0e0e0", edgecolor="white", linewidth=0.8)


# Layer 2: KDE Heatmap (Sightings Gradient)
sns.kdeplot(
    x=us_gdf.geometry.x,
    y=us_gdf.geometry.y,
    cmap="YlOrRd",  # Yellow-Orange-Red gradient
    fill=True,
    alpha=0.6,
    levels=15,
    ax=ax,
)

# Layer 4: Individual Sighting Points
us_gdf.plot(
    ax=ax,
    color="black",
    markersize=8,
    alpha=0.7,
    label="Individual Sightings",
)

# Zoom to Continental US (CONUS)
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

ax.set_title(
    f"US Sightings Heatmap (Total CONUS Reports: {len(us_gdf)})",
    fontsize=14,
    fontweight="bold",
)
ax.set_axis_off()
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

## US Map 2: Heatmap KDE Density Plot

In [ ]:
# Define File Paths for Files inside /geo_maps/us
us_dir = Path("../geo_maps/us")

# Paths to the US State base map & US State shapefiles
us_states_path = us_dir / "tl_2023_us_state.shp"

# Load both shapefile layers
us_states = gpd.read_file(us_states_path)

# Drop AK, HI, and territories so the base map is CONUS only
non_conus = {"02", "15", "60", "66", "69", "72", "78"}
us_states = us_states[~us_states["STATEFP"].isin(non_conus)]

# Build Sighting GeoDataFrame & Align CRS
gdf = gpd.GeoDataFrame(
    df_eda_geocoded,
    geometry=gpd.points_from_xy(df_eda_geocoded["longitude"], df_eda_geocoded["latitude"]),
    crs="EPSG:4326",  # Set Standard Coordinate Reference System (CRS) w/ lat/lon
)

# Align CRS for the shapefile to match sightings data
us_states = us_states.to_crs(gdf.crs)

# Filter sightings to points that actually fall inside a CONUS state polygon.
# The old bounding box (cx[-125:-66, 24:50]) kept ocean and Canadian points.
us_gdf = gpd.sjoin(gdf, us_states[["geometry"]], how="inner", predicate="within")
us_gdf = us_gdf.drop(columns="index_right")

# Report how many points the bbox was letting through
print(f"bbox count: {len(gdf.cx[-125:-66, 24:50])}, within-CONUS count: {len(us_gdf)}")

# Build Layered US Map
plt.close("all")  # Prevent plot memory overlap
fig, ax = plt.subplots(figsize=(14, 10))

# Layer 1: US State Boundaries Base Map (Light gray fill w/ white state borders)
us_states.plot(ax=ax, color="#e0e0e0", edgecolor="white", linewidth=0.8)

# Layer 2: KDE Heatmap (Sightings Gradient)
sns.kdeplot(
    x=us_gdf.geometry.x,
    y=us_gdf.geometry.y,
    cmap="YlOrRd",  # Yellow-Orange-Red gradient
    fill=True,
    alpha=0.6,
    levels=15,
    bw_adjust=0.35,  # tighter kernel; 1.0 was oversmoothing into the ocean
    thresh=0.05,  # drop the faint outer tail entirely
    norm=LogNorm(),
    cbar=True,
    cbar_kws={
        "label": "Estimated report density",
        "shrink": 0.55,
        "orientation": "horizontal",
        "pad": 0.02,
        "format": "%.4f",
    },
    ax=ax,
)

# Layer 3: State boundaries redrawn on top so the KDE fill does not bury them
us_states.boundary.plot(ax=ax, color="#666666", linewidth=0.6, zorder=3)

# Layer 4: Individual Sighting Points
us_gdf.plot(
    ax=ax,
    color="black",
    markersize=8,
    alpha=0.7,
    label="Individual Sightings",
    zorder=4,
)

# Zoom to Continental US (CONUS)
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

ax.set_title(
    f"US Sightings Heatmap (Total CONUS Reports: {len(us_gdf)})",
    fontsize=14,
    fontweight="bold",
)
ax.set_axis_off()
ax.legend(loc="lower right")

plt.tight_layout()
N = len(us_gdf)
cax = fig.axes[-1]                      # the colorbar axis
ticks = cax.get_xticks()
cax.set_xticks(ticks)
cax.set_xticklabels([f"{t * N:.0f}" for t in ticks])
cax.set_xlabel("Reports per square degree")
plt.show()

In [ ]:
print([c for c in gdf.columns if gdf[c].dtype == "object"][:40])

In [ ]:
bbox_gdf = gdf.cx[-125:-66, 24:50]
dropped = bbox_gdf.loc[~bbox_gdf.index.isin(us_gdf.index)]
print(dropped[["City", "State", "latitude", "longitude"]].to_string())

In [ ]:
# Map STUSPS -> geometry for all 50 states + DC
state_geom = us_states.set_index("STUSPS")["geometry"]

check = gdf[gdf["State"].isin(state_geom.index)].copy()

def in_named_state(row):
    poly = state_geom.get(row["State"])
    return poly.contains(row.geometry) if poly is not None else None

check["matches_state"] = check.apply(in_named_state, axis=1)

bad = check[check["matches_state"] == False]
print(f"US-state rows checked: {len(check)}")
print(f"coords outside their own named state: {len(bad)} "
      f"({100*len(bad)/len(check):.1f}%)")

# Which city names are worst
print(bad.groupby(["City", "State"]).size().sort_values(ascending=False).head(40))

In [ ]:
g = gdf[(gdf["City"] == "Garrettsville") & (gdf["State"] == "OH")]
print(g[["City", "State", "latitude", "longitude"]].head())

oh = state_geom.get("OH")
pt = g.geometry.iloc[0]
print("point:", pt)
print("OH bounds:", oh.bounds)
print("contains:", oh.contains(pt))
print("gdf crs:", gdf.crs, "| states crs:", us_states.crs)

In [ ]:
n_nan = gdf[["latitude", "longitude"]].isna().any(axis=1).sum()
print(f"total rows: {len(gdf)}")
print(f"missing coords: {n_nan} ({100*n_nan/len(gdf):.1f}%)")

# rerun the state check on geocoded rows only
check2 = gdf[
    gdf["State"].isin(state_geom.index)
    & gdf["latitude"].notna()
    & gdf["longitude"].notna()
].copy()

check2["matches_state"] = check2.apply(in_named_state, axis=1)
bad2 = check2[check2["matches_state"] == False]

print(f"geocoded US-state rows: {len(check2)}")
print(f"outside their own state: {len(bad2)} ({100*len(bad2)/len(check2):.1f}%)")
print(bad2.groupby(["City", "State"]).size().sort_values(ascending=False).head(30))

In [ ]:
gdf["has_coords"] = gdf["latitude"].notna() & gdf["longitude"].notna()
print(gdf.groupby("occurred_year")["has_coords"].agg(["size", "mean"]).tail(15))

In [ ]:
miss = gdf[~gdf["has_coords"]]
print(miss["State"].isna().mean(), "missing state")
print(miss["City"].isna().mean(), "missing city")
print(miss["City"].value_counts().head(15))

## US Map 3: Chloropleth & Sighting Density Plot

In [ ]:
# Define File Paths for Files inside /geo_maps/us
us_dir = Path("../geo_maps/us")
us_states_path = us_dir / "tl_2023_us_state.shp"

us_states = gpd.read_file(us_states_path)

# Build Sighting GeoDataFrame & Align CRS
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",  # Set Standard Coordinate Reference System (CRS) w/ lat/lon
)

# Align CRS
us_states = us_states.to_crs(gdf.crs)

# Filter sighting data down to Continental US (CONUS)
us_gdf = gdf.cx[-125:-66, 24:50]

# Spatial Join & Aggregate Counts per State
# Join points into state polygons
joined = gpd.sjoin(us_states, us_gdf, how="left", predicate="contains")

# Group by state identifier (STUSPS is two-letter abbreviation, or rely on GEOID)
sighting_counts = (
    joined.groupby("STUSPS").size().reset_index(name="sighting_count")
)

# Merge counts back to original state polygons
us_states = us_states.merge(sighting_counts, on="STUSPS", how="left")
us_states["sighting_count"] = us_states["sighting_count"].fillna(0)

# Build Choropleth Map
plt.close("all")
fig, ax = plt.subplots(figsize=(14, 10))

# Layer 1: Choropleth filled by state sighting counts
us_states.plot(
    column="sighting_count",
    cmap="YlOrRd",  # Yellow-Orange-Red gradient
    linewidth=0.8,
    ax=ax,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Total Sightings per State",
        "orientation": "horizontal",
        "shrink": 0.6,
        "pad": 0.02,
    },
)

# Layer 2: Individual Sighting Points (Optional: lower alpha for clarity)
us_gdf.plot(
    ax=ax,
    color="black",
    markersize=4,
    alpha=0.4,
    label="Individual Sightings",
)

# Framing & Formatting (CONUS Zoom)
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

ax.set_title(
    f"US State Choropleth & Sighting Density (Total CONUS Reports: {len(us_gdf)})",
    fontsize=14,
    fontweight="bold",
)
ax.set_axis_off()
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()